In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import os
from collections import defaultdict
import math

# Importar módulos atualizados (Assumindo que mm_dataset_mealey.py foi atualizado)
from mm_config_mealey import CONFIG, generate_splits, DATA_ROOT, SPLIT_DIR
from mm_dataset_mealey import MMDataset 
# Assumindo que o arquivo mm_transformer_mealey.py está no mesmo diretório
from mm_transformer_mealey import MultiModalTransformer

# --- CONFIGURAÇÃO DE ESTABILIDADE (AJUSTE NOVO) ---
# Limite máximo da norma do gradiente para evitar gradientes explosivos (NAN) na GRU
MAX_GRADIENT_NORM = 1.0 # Valor típico para modelos recorrentes

# --- 0. PREPARAÇÃO INICIAL ---

# Configura o device (GPU ou CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

print("Verificando e gerando arquivos de split...")
# Garantimos que os splits existam antes de carregar
if not os.path.exists(CONFIG["train_split_path"]):
    generate_splits(DATA_ROOT, SPLIT_DIR)


# --- 1. FUNÇÕES DE TREINAMENTO E VALIDAÇÃO ---

def train_epoch(model, dataloader, criterion_pos, criterion_cls, criterion_state, optimizer):
    """
    Executa uma época de treinamento com Backpropagation Through Time (BPTT).
    Assumimos que o batch tem as chaves 'gt_state' e 'gt_drone_class' (ver seção 2 da análise).
    """
    model.train()
    running_loss = 0.0
    num_batches = len(dataloader)
    
    for batch in tqdm(dataloader, desc="Treinamento"):
        
        # O lote agora tem o formato [B, T, ...]
        B, T = batch['image'].shape[:2] 
        
        # Inicializa o estado GRU (Mealy) para o início de CADA sequência no batch (t=0)
        prev_mealy_state = None 
        
        # Listas para coletar as predições de todos os T passos
        all_pred_pos = []
        all_pred_class = []
        all_pred_state_logits = []

        optimizer.zero_grad()
        
        # LOOP PRINCIPAL DO BPTT (Iteração sobre o tempo T)
        for t in range(T):
            
            # 1. Extrai o frame t de todas as modalidades e move para o DEVICE
            image_t = batch['image'][:, t, ...].to(DEVICE)
            lidar_t = batch['lidar'][:, t, ...].to(DEVICE)
            radar_t = batch['radar'][:, t, ...].to(DEVICE)
            audio_t = batch['audio'][:, t, ...].to(DEVICE)
            
            # 2. Forward pass para o passo t (passando o estado anterior)
            output = model(image_t, lidar_t, radar_t, audio_t, prev_mealy_state)
            
            # 3. Coleta predições e estado
            all_pred_pos.append(output['pred_pos'])
            all_pred_class.append(output['pred_class'])
            all_pred_state_logits.append(output['pred_state_logits'])
            
            # 4. Propaga o estado: O estado atual se torna o estado anterior para t+1
            # Importante para BPTT
            prev_mealy_state = output['next_mealy_state'] 
            
        # --- Cálculo da Loss (Empilhando as predições de T) ---
        
        # Predições: [T, B, dim] -> [B*T, dim]
        pred_pos_flat = torch.cat(all_pred_pos, dim=0)
        pred_class_flat = torch.cat(all_pred_class, dim=0)
        pred_state_logits_flat = torch.cat(all_pred_state_logits, dim=0)
        
        # Ground Truths: [B, T, dim] -> [B*T, dim]
        gt_pos_flat = batch['gt_pos'].reshape(B * T, -1).to(DEVICE)
        
        # GT do ESTADO SIMBÓLICO (0, 1, 2, -1) - O QUE É GERADO PELA LÓGICA MEALY
        gt_state_flat = batch.get('gt_state', batch['gt_class']).reshape(B * T).to(DEVICE) 
        
        # GT da CLASSE DO DRONE (0, 1, 2, 3, -1) - O GT RAW DE CLASSIFICAÇÃO
        # Assumimos que o dataloader deve prover uma chave separada
        # Se não houver, voltamos para o gt_state, mas o modelo não aprenderá a classe
        gt_drone_class_flat = batch.get('gt_drone_class', gt_state_flat).reshape(B * T).to(DEVICE) 

        # 1. Loss de Posição (MSE)
        loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
        
        # 2. Loss de Estado Simbólico (Mealy State)
        # Usa gt_state_flat (0, 1, 2, -1) para treinar pred_state_logits
        valid_state_mask = gt_state_flat != -1
        if valid_state_mask.any():
            valid_gt_state = gt_state_flat[valid_state_mask]
            valid_pred_state_logits = pred_state_logits_flat[valid_state_mask]
            loss_state = criterion_state(valid_pred_state_logits, valid_gt_state)
        else:
            loss_state = torch.tensor(0.0).to(DEVICE)

        # 3. Loss de Classificação do Drone (Tipo 0, 1, 2, 3)
        # Usa gt_drone_class_flat (0, 1, 2, 3, -1) para treinar pred_class
        valid_cls_mask = gt_drone_class_flat != -1
        if valid_cls_mask.any():
            valid_gt_class = gt_drone_class_flat[valid_cls_mask]
            valid_pred_class = pred_class_flat[valid_cls_mask]
            loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
        else:
            loss_cls = torch.tensor(0.0).to(DEVICE)
        
        # Loss Total: A soma das 3 cabeças.
        # Você pode adicionar pesos de loss aqui, se desejar (e.g., total_loss = loss_pos + 0.5 * loss_cls + 2.0 * loss_state)
        total_loss = loss_pos + loss_cls + loss_state
        
        # Backward pass
        total_loss.backward()
        
        # IMPLEMENTAÇÃO DO GRADIENT CLIPPING
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRADIENT_NORM)
        
        # Otimização
        optimizer.step()
        
        # running_loss deve ser normalizado pela quantidade de amostras no lote, não apenas pelo lote
        running_loss += total_loss.item() * (B * T) 

    # Retorna a média da loss por amostra (total de amostras = len(dataloader.dataset) * sequence_length)
    total_samples = len(dataloader.dataset) * T
    return running_loss / total_samples


def validate_epoch(model, dataloader, criterion_pos, criterion_cls, criterion_state):
    """
    Executa uma época de validação com BPTT.
    """
    model.eval()
    val_loss_pos = 0.0
    val_loss_cls = 0.0
    val_loss_state = 0.0
    correct_cls_predictions = 0
    total_cls_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validação"):
            B, T = batch['image'].shape[:2]
            
            prev_mealy_state = None 
            
            all_pred_pos = []
            all_pred_class = []
            all_pred_state_logits = []

            for t in range(T):
                # Extrai frame t
                image_t = batch['image'][:, t, ...].to(DEVICE)
                lidar_t = batch['lidar'][:, t, ...].to(DEVICE)
                radar_t = batch['radar'][:, t, ...].to(DEVICE)
                audio_t = batch['audio'][:, t, ...].to(DEVICE)
                
                # Forward pass (passando o estado anterior)
                output = model(image_t, lidar_t, radar_t, audio_t, prev_mealy_state)
                
                all_pred_pos.append(output['pred_pos'])
                all_pred_class.append(output['pred_class'])
                all_pred_state_logits.append(output['pred_state_logits'])
                
                # Propaga o estado
                prev_mealy_state = output['next_mealy_state']
            
            # --- Cálculo da Loss (Empilhando as predições de T) ---
            pred_pos_flat = torch.cat(all_pred_pos, dim=0)
            pred_class_flat = torch.cat(all_pred_class, dim=0)
            pred_state_logits_flat = torch.cat(all_pred_state_logits, dim=0)
            
            gt_pos_flat = batch['gt_pos'].reshape(B * T, -1).to(DEVICE)
            gt_state_flat = batch.get('gt_state', batch['gt_class']).reshape(B * T).to(DEVICE)
            gt_drone_class_flat = batch.get('gt_drone_class', gt_state_flat).reshape(B * T).to(DEVICE)

            # 1. Loss de Posição (MSE)
            loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
            val_loss_pos += loss_pos.item() * (B * T)

            # 2. Loss de Estado Simbólico (Mealy State)
            valid_state_mask = gt_state_flat != -1
            if valid_state_mask.any():
                valid_gt_state = gt_state_flat[valid_state_mask]
                valid_pred_state_logits = pred_state_logits_flat[valid_state_mask]
                loss_state = criterion_state(valid_pred_state_logits, valid_gt_state)
                val_loss_state += loss_state.item() * valid_gt_state.size(0)

            # 3. Loss de Classificação (Tipo do Drone + Accuracy)
            valid_cls_mask = gt_drone_class_flat != -1
            if valid_cls_mask.any():
                valid_gt_class = gt_drone_class_flat[valid_cls_mask]
                valid_pred_class = pred_class_flat[valid_cls_mask]
                
                loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
                val_loss_cls += loss_cls.item() * valid_gt_class.size(0)

                # Accuracy de Classificação do Drone
                _, predicted_classes = torch.max(valid_pred_class, 1)
                correct_cls_predictions += (predicted_classes == valid_gt_class).sum().item()
                total_cls_samples += valid_gt_class.size(0)
            
    # Métricas de Retorno
    total_samples_time = len(dataloader.dataset) * T
    total_valid_state_samples = total_samples_time - (gt_state_flat == -1).sum().item()
    
    # Perda média por amostra
    avg_loss_pos = val_loss_pos / total_samples_time
    avg_loss_cls = val_loss_cls / (total_cls_samples if total_cls_samples > 0 else 1)
    avg_loss_state = val_loss_state / (total_valid_state_samples if total_valid_state_samples > 0 else 1)
    
    accuracy_cls = (correct_cls_predictions / total_cls_samples) * 100 if total_cls_samples > 0 else 0.0
    
    return avg_loss_pos, avg_loss_cls, avg_loss_state, accuracy_cls


# --- 2. CONFIGURAÇÃO PRINCIPAL E EXECUÇÃO ---

def main():
    
    # 2.1. Carregamento dos Dados
    train_dataset = MMDataset(CONFIG["train_split_path"], CONFIG["data_root"], CONFIG, phase='train')
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG["batch_size"], 
        shuffle=True, 
        num_workers=CONFIG["num_workers"],
        pin_memory=True 
    )
    
    # Para o val, é importante usar o mesmo data_root, mas o split de validação
    val_dataset = MMDataset(CONFIG["val_split_path"], CONFIG["data_root"], CONFIG, phase='val') 
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG["batch_size"], 
        shuffle=False, 
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    # 2.2. Inicialização do Modelo, Otimizador e Losses
    
    # Garante que o dropout está no CONFIG para o modelo, se necessário
    if 'dropout_rate' not in CONFIG:
        CONFIG['dropout_rate'] = 0.1 
    
    model = MultiModalTransformer(CONFIG).to(DEVICE)
    
    # Lembre-se: nn.CrossEntropyLoss lida automaticamente com logits
    criterion_pos = nn.MSELoss() 
    # Usado para TIPO DE DRONE (4 classes: 0, 1, 2, 3)
    criterion_cls = nn.CrossEntropyLoss(ignore_index=-1) 
    # Usado para ESTADO SIMBÓLICO (3 classes: 0, 1, 2)
    criterion_state = nn.CrossEntropyLoss(ignore_index=-1) 
    
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    
    # 2.3. Loop Principal de Treinamento
    
    best_val_loss = float('inf')
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\n--- Época {epoch+1}/{CONFIG['epochs']} ---")
        
        # Treinamento
        train_loss = train_epoch(model, train_loader, criterion_pos, criterion_cls, criterion_state, optimizer)
        print(f"Média da Loss de Treinamento (Total): {train_loss:.4f}")
        
        # Validação
        val_loss_pos, val_loss_cls, val_loss_state, val_accuracy_cls = validate_epoch(model, val_loader, criterion_pos, criterion_cls, criterion_state)
        
        total_val_loss = val_loss_pos + val_loss_cls + val_loss_state
        
        print(f"\n[Validação] Loss Total: {total_val_loss:.4f}")
        print(f"  > Loss Posição (MSE): {val_loss_pos:.4f}")
        print(f"  > Loss Classificação Drone: {val_loss_cls:.4f}")
        print(f"  > Loss Estado Simbólico (Mealy): {val_loss_state:.4f}")
        print(f"  > Accuracy Classificação Drone: {val_accuracy_cls:.2f}%")
        
        # Salva o melhor modelo (Critério: Menor Total Loss de Validação)
        if total_val_loss < best_val_loss:
            best_val_loss = total_val_loss
            torch.save(model.state_dict(), 'best_mm_transformer_mealey_model.pth')
            print(">>> Modelo salvo: Melhor resultado de Validação <<<")

    print("\nTreinamento concluído.")

if __name__ == "__main__":
    # Certifique-se de que o DATA_ROOT está configurado corretamente em mm_config.py
    main()
